# 01. 노이즈 채널 베이스라인
`y* = argmax_y  log P_LM(y) + log P_ch(x|y)`
- 채널: 자모(초/중/종성) 분해 + 초성 ㅇ의 연음 조건부
- LM: train output 문자 5-gram (Kneser-Ney)
- 디코딩: 글자별 후보 격자 빔 서치 (출력 길이 = 입력 길이)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/데이콘/난독화
!nproc

## 1) 빠른 동작 확인 (검증 200행)

In [ ]:
!python noisy_channel.py --mode val --val_limit 200

## 2) 전체 검증 (train 90% / val 10%)

In [ ]:
!python noisy_channel.py --mode val

### 채널 효과 비교 (검증 전체)
- `--no_pair` : 예전 독립 채널 (받침 평균 분포 + 연음 조건부 초성)
- 기본 : 받침·연음 결합 채널 + 연음 없는 위치 전용 받침 분포
사전 효과와 섞이지 않도록 `dict_weight 0` 과 `3` 두 경우 모두 비교.

In [ ]:
for dw in [0, 3]:
    for flag in ['--no_pair', '']:
        print(f'=== dict_weight {dw} | channel {"independent" if flag else "pair"}')
        !python noisy_channel.py --mode val --dict_weight {dw} {flag} | grep -E 'noisy-channel'

## 3) 하이퍼파라미터 탐색 → `outputs/best_params.json`
- 모든 실행이 `outputs/nc_runs.csv` 에 (파라미터, F1, 시간) 으로 누적 기록됨
- 기본 탐색: `order 5,6 × lm_weight 1.1,1.3,1.5 × beam 8,16 × dict_oov 0.5,1,2` = 36 조합 (검증 전체)
- 범위 변경: `--grid 'lm_weight=1.2,1.4;dict_weight=2,4'`

In [ ]:
!python noisy_channel.py --mode sweep

In [ ]:
import pandas as pd
r = pd.read_csv('outputs/nc_runs.csv')
r.sort_values('f1', ascending=False).head(15)

## 4) 오류 분석

In [ ]:
import pandas as pd
from collections import Counter
v = pd.read_csv('outputs/val_pred.csv')
print('mean F1', v.f1.mean())
err = Counter()
for g, p in zip(v.output, v.pred):
    for a, b in zip(g, p):
        if a != b:
            err[(a, b)] += 1
print('가장 흔한 오류 (정답, 예측):', err.most_common(30))
for _, r in v.sort_values('f1').head(5).iterrows():
    print('IN  :', r.input[:120]); print('GT  :', r.output[:120]); print('PRED:', r.pred[:120]); print()

## 5) 전체 train으로 학습 → test 예측 / 제출 파일 생성
`best_params.json` 이 있으면 그 설정을 사용하고, 사용한 값은 `submission_noisy_channel_params.json` 에 저장.

In [ ]:
!python noisy_channel.py --mode test   # best_params.json 자동 사용
pd.read_csv('outputs/submission_noisy_channel.csv').head()